In [1]:
import os, re, zipfile, cv2
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import skew, kurtosis
from scipy.fft import fft

print("TensorFlow:", tf.__version__)


TensorFlow: 2.19.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

BASE = "/content/dataset/Parkinson's Dataset Final"
print("Dataset ready at:", BASE)


Mounted at /content/drive
Dataset ready at: /content/dataset/Parkinson's Dataset Final


In [3]:
records = []

def scan_images(base_dir, label):
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".png", ".jpeg")):
                records.append({
                    "path": os.path.join(root, f),
                    "label": label
                })

scan_images(os.path.join(BASE, "New hand pd Dataset", "Parkinson's"), 1)
scan_images(os.path.join(BASE, "New hand pd Dataset", "Healthy"), 0)

image_df = pd.DataFrame(records)

print("Total images:", len(image_df))
print(image_df["label"].value_counts())


Total images: 512
label
1    279
0    233
Name: count, dtype: int64


In [4]:
def extract_pid(path):
    m = re.search(r'P\d+', path)
    return m.group() if m else None

image_df["pid"] = image_df["path"].apply(extract_pid)
image_df.head()


,path,label,pid
0,/content/dataset/Parkinson's Dataset Final/New...,1,P16
1,/content/dataset/Parkinson's Dataset Final/New...,1,P24
2,/content/dataset/Parkinson's Dataset Final/New...,1,P14
3,/content/dataset/Parkinson's Dataset Final/New...,1,P29
4,/content/dataset/Parkinson's Dataset Final/New...,1,P26


In [5]:
signal_root = os.path.join(
    BASE,
    "New hand pd Dataset",
    "Parkinson's",
    "PatientSignal",
    "Signal"
)

signal_map = {}

for root, _, files in os.walk(signal_root):
    for f in files:
        if f.endswith(".txt"):
            m = re.search(r'P\d+', f)
            if m:
                signal_map[m.group()] = os.path.join(root, f)

print("Total signal files:", len(signal_map))


Total signal files: 31


In [6]:
image_df_signal = image_df[image_df["pid"].isin(signal_map.keys())]

print("Images with signals:", len(image_df_signal))
print(image_df_signal["label"].value_counts())


Images with signals: 306
label
1    279
0     27
Name: count, dtype: int64


In [7]:
patient_df = (
    image_df_signal
    .groupby("pid")
    .agg({"label": "first"})
    .reset_index()
)

train_pids, test_pids = train_test_split(
    patient_df["pid"],
    test_size=0.2,
    stratify=patient_df["label"],
    random_state=42
)

train_df = image_df_signal[image_df_signal["pid"].isin(train_pids)]
test_df  = image_df_signal[image_df_signal["pid"].isin(test_pids)]

print("Train:", train_df["label"].value_counts())
print("Test :", test_df["label"].value_counts())


Train: label
1    216
0     22
Name: count, dtype: int64
Test : label
1    63
0     5
Name: count, dtype: int64


In [8]:
IMG_SIZE = 224

def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = img.astype(np.float32) / 255.0
    return img[..., np.newaxis]


In [9]:
def read_signal_file(path):
    data = []
    with open(path) as f:
        for line in f:
            if line.startswith("#") or "<" in line:
                continue
            try:
                vals = [float(x) for x in line.split()]
                data.append(vals)
            except:
                pass
    return pd.DataFrame(data)

def extract_signal_features(df):
    feats = []
    for col in df.columns:
        x = df[col].values
        feats.extend([
            np.mean(x), np.std(x), np.min(x), np.max(x),
            np.median(x), skew(x), kurtosis(x)
        ])
        f = np.abs(fft(x))
        feats.extend([np.mean(f), np.std(f), np.max(f)])
    return np.array(feats, dtype=np.float32)

def load_signal_from_pid(pid):
    df = read_signal_file(signal_map[pid])
    return extract_signal_features(df)


In [10]:
X_img_train = np.array([load_image(p) for p in train_df["path"]])
X_img_test  = np.array([load_image(p) for p in test_df["path"]])

X_sig_train = np.array([load_signal_from_pid(p) for p in train_df["pid"]])
X_sig_test  = np.array([load_signal_from_pid(p) for p in test_df["pid"]])

y_train = train_df["label"].values
y_test  = test_df["label"].values

scaler = StandardScaler()
X_sig_train = scaler.fit_transform(X_sig_train)
X_sig_test  = scaler.transform(X_sig_test)

print(X_img_train.shape, X_sig_train.shape)


(238, 224, 224, 1) (238, 60)


In [11]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = {0: class_weights[0], 1: class_weights[1]}
class_weights


{0: np.float64(5.409090909090909), 1: np.float64(0.5509259259259259)}

In [12]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size=16, embed_dim=526):
        super().__init__()
        self.proj = layers.Conv2D(embed_dim, patch_size, patch_size)

    def call(self, x):
        x = self.proj(x)
        return tf.reshape(x, (tf.shape(x)[0], -1, x.shape[-1]))

class PositionalEncoding(layers.Layer):
    def __init__(self, n, d):
        super().__init__()
        self.pos = layers.Embedding(n, d)

    def call(self, x):
        return x + self.pos(tf.range(tf.shape(x)[1]))

class TransformerBlock(layers.Layer):
    def __init__(self, d):
        super().__init__()
        self.attn = layers.MultiHeadAttention(
            num_heads=4,
            key_dim=d//4,
            output_shape=d
        )
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()
        self.mlp = layers.Dense(d, activation="gelu")
        self.proj = None
        self.d = d

    def build(self, s):
        if s[-1] != self.d:
            self.proj = layers.Dense(self.d)

    def call(self, x):
        sc = self.proj(x) if self.proj else x
        x = self.norm1(sc + self.attn(x, x))
        return self.norm2(x + self.mlp(x))


In [13]:
def build_custom_vit_extractor():
    inp = layers.Input((224,224,1))
    x = PatchEmbedding(16,526)(inp)
    x = PositionalEncoding(196,526)(x)
    for d in [526,256,128,64,32,16,8,4]:
        x = TransformerBlock(d)(x)
    x = layers.GlobalAveragePooling1D()(x)
    return models.Model(inp, x)

vit_extractor = build_custom_vit_extractor()
vit_extractor.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding                 │ (None, None, 526)      │       135,182 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding             │ (None, None, 526)      │       103,096 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, None, 526)      │     1,383,900 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, None, 256)      │       672,256 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, None, 128)      │       165,120 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, None, 64)       │        41,600 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_4             │ (None, None, 32)       │        10,560 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_5             │ (None, None, 16)       │         2,720 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_6             │ (None, None, 8)        │           720 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_7             │ (None, None, 4)        │           200 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 4)              │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,515,354 (9.60 MB)

 Trainable params: 2,515,354 (9.60 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
def build_signal_extractor(d):
    inp = layers.Input((d,))
    x = layers.Dense(128, activation="relu")(inp)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dense(16, activation="relu")(x)
    return models.Model(inp, x)

signal_extractor = build_signal_extractor(X_sig_train.shape[1])


In [15]:
img_in = vit_extractor.input
sig_in = signal_extractor.input

img_f = vit_extractor(img_in)
sig_f = signal_extractor(sig_in)

x = layers.Concatenate()([img_f, sig_f])

for d in [526,256,128,64,32,16,8,4,2]:
    x = layers.Dense(d, activation="relu")(x)

out = layers.Dense(1, activation="sigmoid")(x)

final_model = models.Model([img_in, sig_in], out)
final_model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 60)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 4)         │  2,515,354 │ input_layer[0][0] │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ (None, 16)        │     18,672 │ input_layer_1[0]… │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 20)        │          0 │ functional[0][0], │
│ (Concatenate)       │                   │            │ functional_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 526)       │     11,046 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 256)       │    134,912 │ dense_19[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 128)       │     32,896 │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 64)        │      8,256 │ dense_21[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 32)        │      2,080 │ dense_22[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_24 (Dense)    │ (None, 16)        │        528 │ dense_23[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 8)         │        136 │ dense_24[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 4)         │         36 │ dense_25[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 2)         │         10 │ dense_26[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 1)         │          3 │ dense_27[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,723,929 (10.39 MB)

 Trainable params: 2,723,929 (10.39 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
final_model.compile(
    optimizer=Adam(3e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

history = final_model.fit(
    [X_img_train, X_sig_train],
    y_train,
    validation_data=([X_img_test, X_sig_test], y_test),
    epochs=50,
    batch_size=8,
    class_weight=class_weights
)


Epoch 1/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.3500 - auc: 0.5000 - loss: 0.8153 - val_accuracy: 0.0735 - val_auc: 0.5000 - val_loss: 0.6942
Epoch 2/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.1190 - auc: 0.5000 - loss: 0.7824 - val_accuracy: 0.0735 - val_auc: 0.5000 - val_loss: 0.6940
Epoch 3/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.1271 - auc: 0.5000 - loss: 0.8098 - val_accuracy: 0.0735 - val_auc: 0.5000 - val_loss: 0.6940
Epoch 4/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.0700 - auc: 0.5000 - loss: 0.6177 - val_accuracy: 0.0735 - val_auc: 0.5000 - val_loss: 0.6933
Epoch 5/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.0773 - auc: 0.4677 - loss: 0.6423 - val_accuracy: 0.0735 - val_auc: 0.5000 - val_loss: 0.6938
Epoch 6/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.0990 - auc: 0.5000 - loss: 0.7152 - val_accuracy: 0.0735 - val_auc: 0.5000 - val_loss: 0.6936
Epoch 7/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/s

In [17]:
print("Total images:", len(image_df))
print(image_df["label"].value_counts())


Total images: 512
label
1    279
0    233
Name: count, dtype: int64


In [18]:
patient_df = (
    image_df
    .groupby("pid")
    .agg({"label": "first"})
    .reset_index()
)

train_pids, test_pids = train_test_split(
    patient_df["pid"],
    test_size=0.2,
    stratify=patient_df["label"],
    random_state=42
)

train_df = image_df[image_df["pid"].isin(train_pids)]
test_df  = image_df[image_df["pid"].isin(test_pids)]

print("Train balance:\n", train_df["label"].value_counts())
print("Test balance:\n", test_df["label"].value_counts())


Train balance:
 label
1    216
0     27
Name: count, dtype: int64
Test balance:
 label
1    63
0     7
Name: count, dtype: int64


In [19]:
IMG_SIZE = 224
BATCH = 16

augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1)
])

def load_image_tf(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=1)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["path"].values, train_df["label"].values)
)
train_ds = (
    train_ds
    .map(load_image_tf, num_parallel_calls=tf.data.AUTOTUNE)
    .map(lambda x,y: (augment(x), y))
    .shuffle(512)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices(
    (test_df["path"].values, test_df["label"].values)
)
val_ds = (
    val_ds
    .map(load_image_tf, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)


In [20]:
def build_image_vit():
    inp = layers.Input((224,224,1))

    x = layers.Conv2D(64, 7, strides=2, padding="same")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = PatchEmbedding(16, 256)(x)
    x = PositionalEncoding(196, 256)(x)

    for d in [256,128,64,32]:
        x = TransformerBlock(d)(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation="relu")(x)

    out = layers.Dense(1, activation="sigmoid")(x)

    return tf.keras.Model(inp, out)

model = build_image_vit()
model.summary()


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │         3,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding_1               │ (None, None, 256)      │     4,194,560 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_encoding_1           │ (None, None, 256)      │        50,176 │
│ (PositionalEncoding)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_8             │ (None, None, 256)      │       329,984 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_9             │ (None, None, 128)      │       165,120 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_10            │ (None, None, 64)       │        41,600 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_11            │ (None, None, 32)       │        10,560 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 128)            │         4,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,808,001 (18.34 MB)

 Trainable params: 4,807,873 (18.34 MB)

 Non-trainable params: 128 (512.00 B)

In [21]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)
class_weights = {0: class_weights[0], 1: class_weights[1]}

model.compile(
    optimizer=Adam(3e-4),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    class_weight=class_weights
)


Epoch 1/40
